# Part C: Prompt Engineering
**Student:** Dabone Abdoul Latif  
**Index:** 10012200015

In [ ]:
import json
import os
from groq import Groq

# Load data from previous parts
with open('chunks/csv_chunks.json', 'r') as f:
    csv_chunks = json.load(f)
with open('chunks/pdf_chunks.json', 'r') as f:
    pdf_chunks = json.load(f)
all_chunks = csv_chunks + pdf_chunks

client = Groq(api_key="your_api_key_here")

### 1. Prompt Templates

In [ ]:
def prompt_v1(context, query):
    return f"""Use the following context to answer the question.
If the answer is not in the context, say you do not know.

Context:
{context}

Question: {query}
Answer:"""

def prompt_v2(context, query):
    return f"""Answer the question based only on the provided documents.
Be direct and concise.

Documents:
{context}

User Query: {query}
Answer:"""

### 2. Context Management (Truncation)

In [ ]:
def limit_context(chunks, max_chars=4000):
    """Keep context within token limits by truncating text"""
    context = ""
    for c in chunks[:3]: # Use top 3
        context += f"\nSource: {c['source']}\n{c['text']}\n"
    return context[:max_chars]

### 3. Experiments

In [ ]:
def run_test(query, chunks):
    context = limit_context(chunks)
    
    print(f"Testing Query: {query}\n")
    
    for p_func in [prompt_v1, prompt_v2]:
        prompt = p_func(context, query)
        res = client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        print(f"--- {p_func.__name__} Output ---")
        print(res.choices[0].message.content)
        print("\n")